# GHDNN Stage 2 — new evidence experiments (Kaggle GPU)

**What this notebook does:** (1) whole-image frozen GoogLeNet baseline + PCA64 + balanced RBF-SVM, (2) 64-patch PCA dimensionality sensitivity (16, 32, 64, 128 and no PCA), (3) 20 independent random patch-selection seeds under the **same five similarity-group folds**. It also produces out-of-fold predictions and traceable fold records.

**Data needed:** existing Kaggle feature cache from the original study including `features_64_224.npy` (shape 1054×64×1024), `labels.npy`, `groups.npy`, `audit.json`; original CT JPG image ZIP; official `googlenet-1378be20.pth`; and `image_manifest.csv` from the supplied reproducibility package.

**Important:** There are no new results until you execute these cells on the genuine medical data. PCA and the scaler are fitted on development folds only. The primary 17-configuration model selection still needs separate nested CV; this supplementary notebook does not itself correct model-selection optimism.

In [ ]:
from pathlib import Path
import os, glob
print('Kaggle input contents:')
for p in sorted(glob.glob('/kaggle/input/**/*',recursive=True)):
    if Path(p).is_file() and Path(p).suffix.lower() in {'.npy','.json','.csv','.zip','.pth'}:
        print(p)
print('GPU:',__import__('torch').cuda.is_available())

In [ ]:
# UPDATE THE PATHS using those printed in the previous cell
CACHE_DIR = '/kaggle/input/YOUR-FEATURE-CACHE'  # folder with features_64_224.npy, labels.npy, groups.npy, audit.json
MANIFEST  = '/kaggle/input/YOUR-MANIFEST/image_manifest.csv'
CT_ZIP    = '/kaggle/input/YOUR-CT-DATA/iqoth-original.zip'
WEIGHTS   = '/kaggle/input/YOUR-GOOGLENET-WEIGHTS/googlenet-1378be20.pth'
OUT       = '/kaggle/working/GHDNN_stage2_results'
for p in [CACHE_DIR,MANIFEST,CT_ZIP,WEIGHTS]:
    assert Path(p).exists(), f'Edit missing Kaggle path: {p}'


In [ ]:
from pathlib import Path
Path("run_stage2_kaggle.py").write_text('#!/usr/bin/env python3\n"""New evidence only: whole-image baseline, patch PCA sensitivity, multi-seed random baseline.\nInputs MUST include matched 64-patch frozen GoogLeNet features, label & group arrays,\naudit file and SHA-indexed manifest generated from the same original images.\nNo medical results are generated by this code until it is run on actual features.\n"""\nimport os\nos.environ.setdefault(\'OPENBLAS_NUM_THREADS\',\'3\')\nos.environ.setdefault(\'OMP_NUM_THREADS\',\'3\')\nimport json, csv, zipfile, hashlib, argparse, pathlib\nfrom collections import Counter\nimport numpy as np\nfrom sklearn.decomposition import PCA\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import accuracy_score, f1_score, recall_score, balanced_accuracy_score\n\nCLASSES=[\'Benign\',\'Malignant\',\'Normal\']\n\ndef sha_file(path,block=1<<20):\n h=hashlib.sha256()\n with open(path,\'rb\') as f:\n  for b in iter(lambda:f.read(block),b\'\'):\n   h.update(b)\n return h.hexdigest()\n\ndef load_manifest(p):\n with open(p, encoding=\'utf-8-sig\',newline=\'\') as f: r=list(csv.DictReader(f))\n assert [int(x[\'index\']) for x in r]==list(range(len(r))), \'Manifest rows must be in original feature-cache order\'\n return r\n\ndef load_whole_features(args,manifest):\n out=pathlib.Path(args.out);out.mkdir(parents=True,exist_ok=True)\n cache=out/\'whole_image_googlenet_1024.npy\'\n if cache.exists():\n  X=np.load(cache,mmap_mode=\'r\');assert X.shape==(len(manifest),1024)\n  print(\'Using saved whole-image cache:\',cache,flush=True)\n  return X\n if not args.dataset or not args.weights:\n  print(\'Whole-image baseline SKIPPED: provide --dataset and --weights to extract it.\',flush=True)\n  return None\n import cv2, torch\n from torchvision.models import googlenet\n from numpy.lib.format import open_memmap\n weights_sha=sha_file(args.weights)\n if not weights_sha.startswith(\'1378be20\'):\n  raise ValueError(\'The checkpoint does not match official torchvision GoogLeNet weights. Refusing to use it.\')\n model=googlenet(weights=None,aux_logits=True,transform_input=False,init_weights=False)\n model.load_state_dict(torch.load(args.weights,map_location=\'cpu\',weights_only=True), strict=True)\n model.aux1=None;model.aux2=None;model.aux_logits=False\n model.fc=torch.nn.Identity();model.transform_input=True;model.eval()\n device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\');model.to(device)\n print(\'Extracting whole-image deep features on\',device,flush=True)\n mean=torch.tensor([.485,.456,.406],device=device).view(1,3,1,1)\n std=torch.tensor([.229,.224,.225],device=device).view(1,3,1,1)\n data=pathlib.Path(args.dataset)\n if data.is_file() and data.suffix.lower()==\'.zip\':\n  with zipfile.ZipFile(data) as z:\n   img_bytes={hashlib.sha256(z.read(k)).hexdigest():z.read(k) for k in z.namelist() if k.lower().endswith((\'.jpg\',\'.jpeg\',\'.png\'))}\n elif data.is_dir():\n  files=[p for p in data.rglob(\'*\') if p.suffix.lower() in (\'.jpg\',\'.jpeg\',\'.png\')]\n  img_bytes={hashlib.sha256(b).hexdigest():b for p in files for b in [p.read_bytes()]}\n else: raise ValueError(\'Provide folder or ZIP as --dataset\')\n missing=[r[\'sha256\'] for r in manifest if r[\'sha256\'] not in img_bytes]\n if missing: raise ValueError(f\'Missing {len(missing)} manifest images from dataset; do not align features by position without SHA checking.\')\n tmp=cache.with_suffix(\'.partial.npy\');X=open_memmap(tmp,mode=\'w+\',dtype=\'float32\',shape=(len(manifest),1024))\n bs=16\n for i in range(0,len(manifest),bs):\n  ims=[]\n  for r in manifest[i:i+bs]:\n   im=cv2.imdecode(np.frombuffer(img_bytes[r[\'sha256\']],dtype=\'uint8\'),cv2.IMREAD_GRAYSCALE)\n   if im is None: raise RuntimeError(\'Cannot decode \'+r[\'file\'])\n   im=cv2.resize(im,(224,224),interpolation=cv2.INTER_AREA)\n   ims.append(im)\n  tens=torch.from_numpy(np.stack(ims).astype(\'float32\')[:,None]/255.).repeat(1,3,1,1).to(device)\n  tens=(tens-mean)/std\n  with torch.inference_mode():feat=model(tens).detach().cpu().numpy()\n  X[i:i+len(ims)]=feat\n  if (i//bs)%10==0:print(\'Whole-image features:\',i+len(ims),\'/\',len(manifest),flush=True)\n X.flush();del X\n os.replace(tmp,cache)\n return np.load(cache,mmap_mode=\'r\')\n\ndef svc_predict(tr,te,yt):\n clf=make_pipeline(StandardScaler(),SVC(C=1.0,kernel=\'rbf\',gamma=\'scale\',class_weight=\'balanced\'))\n clf.fit(tr,yt)\n return clf.predict(te)\n\ndef metrics(y,p):\n return dict(accuracy_pct=100*accuracy_score(y,p),macro_f1_pct=100*f1_score(y,p,average=\'macro\',labels=[0,1,2],zero_division=0),\n   balanced_accuracy_pct=100*balanced_accuracy_score(y,p), benign_recall_pct=100*recall_score(y,p,labels=[0],average=\'macro\',zero_division=0))\n\ndef evaluate(args):\n out=pathlib.Path(args.out);out.mkdir(exist_ok=True,parents=True)\n root=pathlib.Path(args.cache)\n Z=np.load(root/\'features_64_224.npy\',mmap_mode=\'r\')\n Y=np.load(root/\'labels.npy\');GR=np.load(root/\'groups.npy\')\n manifest=load_manifest(args.manifest)\n if Z.shape!=(len(Y),64,1024) or len(manifest)!=len(Y):\n  raise ValueError(f\'Cache shape {Z.shape}, y {len(Y)}, manifest {len(manifest)} mismatch\')\n ym=np.array([int(r[\'label_code\']) for r in manifest]);gm=np.array([int(r[\'similarity_group\']) for r in manifest])\n if not np.array_equal(Y,ym) or not np.array_equal(GR,gm):\n  raise ValueError(\'Feature cache does not match supplied manifest label/group order\')\n if (root/\'audit.json\').exists():\n  audit=json.loads((root/\'audit.json\').read_text())\n  if audit.get(\'pretrained_sha256\') and args.weights:\n   if audit[\'pretrained_sha256\']!=sha_file(args.weights): raise ValueError(\'GoogLeNet checkpoint hash does not match cached features\')\n  chk=hashlib.sha256((audit[\'pretrained_sha256\']+\'|\'+\'\'.join(r[\'sha256\'] for r in manifest)+\'|224net|224orig|transform_input=True\').encode()).hexdigest()\n  if audit.get(\'cache_fingerprint\')!=chk:\n   raise ValueError(\'The manifest SHA order differs from original cached patch feature extraction; aborting.\')\n else:\n  raise ValueError(\'Missing audit.json from the same Kaggle patch-feature extraction; do not compare unmatched data\')\n whole=load_whole_features(args,manifest)\n cv=list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42).split(np.zeros(len(Y)),Y,GR))\n for fi,(tr,te) in enumerate(cv,start=1):\n  if (set(GR[tr]) & set(GR[te])): raise RuntimeError(\'Similarity group leakage at fold \'+str(fi))\n  expected=np.array([int(r[\'fold\']) for r in manifest])\n  if not np.all(expected[te]==fi):raise RuntimeError(\'Fold assignment differs from manifest\')\n pred={name:np.full(len(Y),-1,dtype=int) for name in [\'pca16\',\'pca32\',\'pca64\',\'pca128\',\'no_pca\']}\n if whole is not None:pred[\'whole_image_pca64\']=np.full(len(Y),-1,dtype=int)\n seed_preds={seed:np.full(len(Y),-1,dtype=int) for seed in range(args.random_seeds)}\n selected_rows=[];fold_rows=[]\n for fi,(tr,te) in enumerate(cv,start=1):\n  print(\'Fold\',fi,\'of 5 | dev\',len(tr),\'test\',len(te),flush=True)\n  a=np.asarray(Z[tr],dtype=np.float32);b=np.asarray(Z[te],dtype=np.float32)\n  yt=Y[tr]\n  for pcs in [16,32,64,128]:\n   print(\'  fitting PCA\',pcs,\'from development patches only\',flush=True)\n   pca=PCA(n_components=pcs,svd_solver=\'randomized\',random_state=2026)\n   pca.fit(a.reshape(-1,1024))\n   za=pca.transform(a.reshape(-1,1024)).reshape(len(tr),64,pcs).astype(\'float32\')\n   zb=pca.transform(b.reshape(-1,1024)).reshape(len(te),64,pcs).astype(\'float32\')\n   pred[f\'pca{pcs}\'][te]=svc_predict(za.reshape(len(tr),-1),zb.reshape(len(te),-1),yt)\n   if pcs==64:\n    for seed in range(args.random_seeds):\n     chosen=sorted(np.random.default_rng(62026+fi+64+20+10000*seed).choice(64,20,replace=False).tolist())\n     xtr=za[:,chosen,:].reshape(len(tr),-1);xte=zb[:,chosen,:].reshape(len(te),-1)\n     seed_preds[seed][te]=svc_predict(xtr,xte,yt)\n     selected_rows.append({\'fold\':fi,\'seed_index\':seed,\'rng_seed\':62026+fi+64+20+10000*seed,\'positions_zero_based\':json.dumps(chosen)})\n   del za,zb,pca\n  print(\'  no-PCA full 64 patches -> 65,536-d SVM\',flush=True)\n  pred[\'no_pca\'][te]=svc_predict(a.reshape(len(tr),-1),b.reshape(len(te),-1),yt)\n  del a,b\n  if whole is not None:\n   pca=PCA(n_components=64,svd_solver=\'randomized\',random_state=2026)\n   xa=pca.fit_transform(np.asarray(whole[tr]));xt=pca.transform(np.asarray(whole[te]))\n   pred[\'whole_image_pca64\'][te]=svc_predict(xa,xt,yt)\n  fold_rows.append({\'fold\':fi,\'train_images\':len(tr),\'test_images\':len(te),\'train_groups\':len(set(GR[tr])),\'test_groups\':len(set(GR[te]))})\n # all analysis is from pooled out-of-fold predictions\n results=[]\n for name,p in pred.items():\n  assert np.all(p>=0),name\n  results.append({\'model\':name,\'condition\':\'fixed group CV / exploratory\',\'n\':len(p),**metrics(Y,p)})\n rand_rows=[]\n for seed,p in seed_preds.items():\n  assert np.all(p>=0)\n  rand_rows.append({\'seed_index\':seed,**metrics(Y,p)})\n if rand_rows:\n  for field in [\'accuracy_pct\',\'macro_f1_pct\',\'benign_recall_pct\']:\n   vals=np.array([r[field] for r in rand_rows]);print(f\'random20 {field}: {vals.mean():.3f}±{vals.std(ddof=1):.3f}\',flush=True)\n  results.append({\'model\':\'random20_multiseed\',\'condition\':f\'{args.random_seeds} random seeds on fixed group CV; mean ±sd across seeds\',\'n\':len(Y),\n    **{k:float(np.mean([r[k] for r in rand_rows])) for k in metrics(Y,Y)}})\n  results[-1][\'macro_f1_sd_pct\']=float(np.std([r[\'macro_f1_pct\'] for r in rand_rows],ddof=1))\n def save(name,rows):\n  if not rows:return\n  keys=list(dict.fromkeys(k for row in rows for k in row));\n  with (out/name).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n   w=csv.DictWriter(f,fieldnames=keys);w.writeheader();w.writerows(rows)\n save(\'stage2_ablation_summary.csv\',results)\n save(\'random20_multiseed_summary.csv\',rand_rows)\n save(\'random20_selected_patch_positions.csv\',selected_rows)\n save(\'stage2_fold_distribution.csv\',fold_rows)\n oof=[]\n for i,r in enumerate(manifest):\n  x={\'index\':i,\'file\':r[\'file\'],\'sha256\':r[\'sha256\'],\'label_code\':int(Y[i]),\'group\':int(GR[i]),\'fold\':r[\'fold\']}\n  for k,p in pred.items(): x[\'pred_\'+k]=int(p[i])\n  for seed,p in seed_preds.items():x[f\'random20_seed{seed:02d}\']=int(p[i])\n  oof.append(x)\n save(\'stage2_oof_predictions.csv\',oof)\n # Verify that the original 64-patch reference result is still reproduced (allow tiny numerical tolerance).\n reference=next(v for v in results if v[\'model\']==\'pca64\')\n if abs(reference[\'accuracy_pct\']-96.68)>0.05 or abs(reference[\'macro_f1_pct\']-93.02)>0.05:\n  print(\'WARNING: reconstructed 64-patch results differ from published reference; CHECK input and preprocessing.\', reference, flush=True)\n (out/\'stage2_run_info.json\').write_text(json.dumps({\'cache\':str(root),\'manifest\':args.manifest,\'n\':len(Y),\'folds\':5,\'random_seeds\':args.random_seeds,\'pca_dims\':[16,32,64,128],\'missing_test\':\'No nested CV run; selected-best-of-17 remains exploratory\',\'full_image_available\':whole is not None,\'patient_disjoint\':False},indent=2))\n print(\'New experiment results saved to\',out,flush=True)\n\nif __name__==\'__main__\':\n ap=argparse.ArgumentParser()\n ap.add_argument(\'--cache\',required=True,help=\'Path containing features_64_224.npy, labels.npy, groups.npy, audit.json\')\n ap.add_argument(\'--manifest\',required=True,help=\'Path to image_manifest.csv\')\n ap.add_argument(\'--dataset\',help=\'Original IQ-OTH/NCCD images folder or ZIP for whole-image baseline\')\n ap.add_argument(\'--weights\',help=\'Official googlenet-1378be20.pth\')\n ap.add_argument(\'--out\',default=\'./GHDNN_stage2_results\')\n ap.add_argument(\'--random-seeds\',type=int,default=20)\n args=ap.parse_args()\n if args.random_seeds<2:raise ValueError(\'Use >=2 random seeds\')\n evaluate(args)\n',encoding="utf-8")
print("Saved experimental runner")

In [ ]:
import sys,subprocess
subprocess.run([sys.executable,'-u','run_stage2_kaggle.py','--cache',CACHE_DIR,'--manifest',MANIFEST,'--dataset',CT_ZIP,'--weights',WEIGHTS,'--out',OUT,'--random-seeds','20'],check=True)

In [ ]:
from pathlib import Path
import pandas as pd
out=Path(OUT)
print(pd.read_csv(out/'stage2_ablation_summary.csv').to_string(index=False))
print(pd.read_csv(out/'random20_multiseed_summary.csv').describe().to_string())
import shutil
shutil.make_archive('/kaggle/working/GHDNN_stage2_result_package','zip',out)
print('Output ZIP saved at /kaggle/working/GHDNN_stage2_result_package.zip')